# DRAFT: reconstructing training data from a trained Random Forest

Install with `python -m pip install -e ".[notebook]"` and open this notebook from the repository root. The saved results below can produce different timings and solver outcomes.


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from utils import * 
from datasets_infos import datasets_ohe_vectors, predictions

# Set parameters
seed = 0
dataset = 'compas'
train_size=100
test_size=100
num_threads = -1 # Use all available threads

In [15]:
# Load data
ohe_vector = datasets_ohe_vectors[dataset]
prediction = predictions[dataset]
X_train, X_test, y_train, y_test = load_dataset(dataset='data/%s.csv' % dataset, label=prediction, train_size=100, test_size=100, seed=seed)

# Display the complete training dataset
training_data = X_train.copy()
training_data[prediction] = y_train
display_dataset(training_data, title=f"{dataset} training set")


Example,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor,recidivate-within-two-years = 1
1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0
3,1,0,0,0,1,0,0,0,0,0,0,1,0,1,1
4,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0
5,1,0,0,0,1,0,0,0,0,0,1,0,0,1,1
6,1,0,1,0,0,0,0,0,0,1,0,0,0,1,1
7,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
8,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1
9,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
10,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0


# Basic example

In [3]:
# Train a Random Forest
clf = RandomForestClassifier(n_estimators=100, max_depth=None, bootstrap=True, random_state=seed)
clf = clf.fit(X_train, y_train)

In [4]:
# Reconstruct the Random Forest's training set
from DRAFT import DRAFT

extractor = DRAFT(clf, one_hot_encoded_groups=ohe_vector)
dict_res = extractor.fit(timeout=60, n_threads=num_threads, seed=seed)

In [5]:
print_reconstruction_results(
    "Basic sklearn RandomForestClassifier",
    dict_res,
    X_train,
    display_reconstruction=True,
)

(Basic sklearn RandomForestClassifier) Complete solving duration : 35.978 seconds.
(Basic sklearn RandomForestClassifier) Reconstruction Error: 0.000000


X_train row,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor
1,1,0,0,0,0,1,0,0,0,0,0,1,0,0
2,1,0,0,0,1,0,0,0,0,1,0,0,0,1
3,1,0,0,0,1,0,0,0,0,0,0,1,0,1
4,1,0,0,0,1,0,0,0,0,0,0,0,1,0
5,1,0,0,0,1,0,0,0,0,0,1,0,0,1
6,1,0,1,0,0,0,0,0,0,1,0,0,0,1
7,1,0,0,1,0,0,0,0,0,0,1,0,0,0
8,1,0,1,0,0,0,0,0,0,1,0,0,0,0
9,0,0,0,0,1,0,0,0,0,0,1,0,0,0
10,1,0,0,0,1,0,0,0,0,0,0,1,0,0


# What about protecting the Random Forest using Differential Privacy ?

In [6]:
from DP_RF import DP_RF

epsilon = 10 # DP Budget

# Train a DP-protected Random Forest
clf_dp = DP_RF(n_estimators=10, max_depth=5, random_state=0)
clf_dp.fit(X_train,y_train)

# Store the unnnoised RF
import copy
clf_nondp = copy.deepcopy(clf_dp)

clf_dp.add_noise(epsilon)

print("DP Random Forest accuracy (train):", accuracy_score(y_train, clf_dp.predict(X_train)))
print("DP Random Forest accuracy (test):", accuracy_score(y_test, clf_dp.predict(X_test)))
print("Non-DP (same) Random Forest accuracy (train):", accuracy_score(y_train, clf_nondp.predict(X_train)))
print("Non-DP Random (same) Forest accuracy (test):", accuracy_score(y_test, clf_nondp.predict(X_test)))


DP Random Forest accuracy (train): 0.68
DP Random Forest accuracy (test): 0.6
Non-DP (same) Random Forest accuracy (train): 0.72
Non-DP Random (same) Forest accuracy (test): 0.64


In [7]:
# Reconstruct the DP-protected Random Forest's training set
from DP_RF_solver import DRAFT_DP

extractor_dp = DRAFT_DP(clf_dp, epsilon, one_hot_encoded_groups=ohe_vector)
dict_res_dp = extractor_dp.fit(train_size, seed=seed, timeout=60, n_threads=num_threads)


In [8]:
# Reconstruct the non-DP Random Forest's training set
from DRAFT import DRAFT

extractor_nondp = DRAFT(clf_nondp, one_hot_encoded_groups=ohe_vector)
dict_res_nondp = extractor_nondp.fit(timeout=60, n_threads=num_threads, seed=seed)

In [9]:
print_reconstruction_results(
    "DP-protected RF",
    dict_res_dp,
    X_train,
    display_reconstruction=True,
)

(DP-protected RF) Complete solving duration : 63.333 seconds.
(DP-protected RF) Reconstruction Error: 0.063571


X_train row,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor
1,1,0,0,0,0,1,0,0,0,0,0,1,0,0
2,1,0,0,0,1,0,0,0,1,1,0,0,0,1
3,1,0,0,0,1,0,1,0,0,0,0,1,0,1
4,1,0,0,0,1,0,0,0,0,0,0,0,1,0
5,1,0,0,0,1,0,0,0,0,0,1,0,0,1
6,0,0,1,0,0,0,0,0,1,1,0,0,0,1
7,0,0,0,1,0,0,0,0,0,0,1,0,0,0
8,1,0,1,0,0,0,0,0,0,1,0,0,0,0
9,0,0,0,0,1,0,0,0,0,0,1,0,0,0
10,1,0,0,0,1,0,0,0,0,0,0,1,0,0


In [10]:
print_reconstruction_results(
    "non-DP-protected RF",
    dict_res_nondp,
    X_train,
    display_reconstruction=True,
)

(non-DP-protected RF) Complete solving duration : 1.491 seconds.
(non-DP-protected RF) Reconstruction Error: 0.010714


X_train row,sex = Male,age = 18-20,age = 21-22,age = 23-25,age = 26-45,age = >45,juvenile-felonies = >0,juvenile-misdemeanors = >0,juvenile-crimes = >0,priors = 2-3,priors = =0,priors = =1,priors = >3,current-charge-degree = Misdemeanor
1,1,0,0,0,0,1,0,0,0,0,0,1,0,0
2,1,0,0,0,1,0,0,0,0,1,0,0,0,1
3,1,0,0,0,1,0,0,0,0,0,0,1,0,1
4,1,0,0,0,1,0,0,0,0,0,0,0,1,0
5,1,0,0,0,1,0,0,0,0,0,1,0,0,1
6,1,0,1,0,0,0,0,0,0,1,0,0,0,1
7,1,0,0,1,0,0,0,0,0,0,1,0,0,0
8,1,0,1,0,0,0,0,0,0,1,0,0,0,0
9,0,0,0,0,1,0,0,0,0,0,1,0,0,0
10,1,0,0,0,1,0,0,0,0,0,0,1,0,0


# Test with another DP budget

In [11]:


epsilons = [5] # Other DP Budget

for epsilon_bis in epsilons:
    clf_dp_bis = copy.deepcopy(clf_nondp) # use the same structure as before, with a different epsilon value

    clf_dp_bis.add_noise(epsilon_bis)

    # Reconstruct the DP-protected Random Forest's training set
    from DP_RF_solver import DRAFT_DP

    extractor_dp_bis = DRAFT_DP(clf_dp_bis, epsilon_bis, one_hot_encoded_groups=ohe_vector)
    dict_res_dp_bis = extractor_dp_bis.fit(train_size, seed=seed, timeout=60, n_threads=num_threads)

    print("%d-DP Random Forest accuracy (train):" %epsilon_bis, accuracy_score(y_train, clf_dp_bis.predict(X_train)))
    print("%d-DP Random Forest accuracy (test):" %epsilon_bis, accuracy_score(y_test, clf_dp_bis.predict(X_test)))
    print_reconstruction_results(
        "%d-DP RF" %epsilon_bis,
        dict_res_dp_bis,
        X_train,
        display_reconstruction=False,
    )   


5-DP Random Forest accuracy (train): 0.69
5-DP Random Forest accuracy (test): 0.61
(5-DP RF) Complete solving duration : 64.421 seconds.
(5-DP RF) Reconstruction Error: 0.187857
